# Exa Web Search Agent

The Exa Web Search Agent is an AI-powered research assistant that searches the live web using [Exa's](https://exa.ai) neural search API. It provides three specialised search tools — general web search, code and documentation lookup, and company research — giving it the ability to answer questions with up-to-date information from across the internet.


In [1]:
!pip install -q aixplain requests

## Add your API keys

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

Get your Exa API key from the [Exa Dashboard](https://dashboard.exa.ai/api-keys).

In [ ]:


AixplainAPIKey = "<YOUR_AIXPLAIN_API_KEY>"  #@param {type:"string"}
ExaApiKey = "<EXA_API_KEY>"           #@param {type:"string"}

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey
os.environ["EXA_API_KEY"] = ExaApiKey


In [ ]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)


# What is the Exa Web Search Agent?

The goal of this agent is to create an AI-powered research assistant that can search the live web to answer questions, find code examples, and gather business intelligence — all in real time.

## Agent Architecture

The agent uses three custom script tools, each wrapping a different call to the Exa search API:

- **Web Search** — general-purpose neural web search for current information, news, and facts.
- **Code Context Search** — targets GitHub, Stack Overflow, and official documentation sites to find code examples and API references.
- **Company Research** — gathers business intelligence and recent news about any organisation.

## Agent Workflow

1. **Understand the request** — determine whether it calls for general search, code lookup, or company research.
2. **Call the right tool** — select the most appropriate search tool based on the query type.
3. **Synthesise results** — summarise the findings, cite sources, and present a clear answer.


In [4]:
def web_search_exa(query: str, num_results: int = 8, search_type: str = "auto") -> str:
    """
    Searches the web using Exa's neural search API for current information, news, or facts.

    Parameters:
        query (str): The search query.
        num_results (int): Number of results to return (default: 8).
        search_type (str): Search speed profile — 'auto' (~1s), 'instant' (~200ms),
                           'fast' (~450ms), or 'deep' (5-60s). Default: 'auto'.

    Returns:
        str: Formatted search results with titles, URLs, and content highlights.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    url = "https://api.exa.ai/search"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "query": query,
        "type": search_type,
        "numResults": max(1, min(num_results, 50)),
        "contents": {
            "highlights": {
                "maxCharacters": 4000,
                "numSentences": 3,
            }
        },
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return "No results found."

    formatted = []
    for i, item in enumerate(results, start=1):
        title = item.get("title", "No title")
        link = item.get("url", "")
        highlights = item.get("highlights", [])
        snippet = " ".join(highlights) if highlights else item.get("text", "")[:400]
        formatted.append(f"[{i}] {title}\nURL: {link}\n{snippet}")

    return "\n\n".join(formatted)


In [5]:
def get_code_context_exa(query: str, num_results: int = 5) -> str:
    """
    Searches for code examples and documentation using Exa's neural search API,
    targeting GitHub and Stack Overflow sources.

    Parameters:
        query (str): The code or API search query, e.g. 'Python async context manager example',
                     'React useEffect cleanup function'.
        num_results (int): Number of results to return (default: 5).

    Returns:
        str: Formatted results with titles, URLs, and code/documentation snippets.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    url = "https://api.exa.ai/search"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "query": query,
        "type": "auto",
        "numResults": max(1, min(num_results, 20)),
        "includeDomains": ["github.com", "stackoverflow.com", "docs.python.org",
                           "developer.mozilla.org", "reactjs.org", "docs.rs"],
        "contents": {
            "text": {
                "maxCharacters": 5000,
            }
        },
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return "No results found."

    formatted = []
    for i, item in enumerate(results, start=1):
        title = item.get("title", "No title")
        link = item.get("url", "")
        text = item.get("text", "")[:600]
        formatted.append(f"[{i}] {title}\nURL: {link}\n{text}")

    return "\n\n".join(formatted)


In [6]:
def company_research_exa(company_name: str, num_results: int = 5) -> str:
    """
    Gathers business intelligence and recent news about a company using Exa's neural search API.

    Parameters:
        company_name (str): The name of the company to research, e.g. 'OpenAI', 'Stripe', 'Notion'.
        num_results (int): Number of results to return (default: 5).

    Returns:
        str: Formatted results with titles, URLs, and summaries of recent news and business information.
    """
    import requests
    import os

    api_key = os.environ.get("EXA_API_KEY")
    if not api_key:
        return "Error: EXA_API_KEY environment variable is not set."

    url = "https://api.exa.ai/search"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": api_key,
    }
    payload = {
        "query": f"{company_name} company news funding product launch",
        "type": "auto",
        "numResults": max(1, min(num_results, 20)),
        "category": "company",
        "contents": {
            "highlights": {
                "maxCharacters": 4000,
                "numSentences": 4,
            }
        },
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}\nResponse: {e.response.text if e.response else 'N/A'}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results = data.get("results", [])
    if not results:
        return f"No results found for {company_name}."

    formatted = []
    for i, item in enumerate(results, start=1):
        title = item.get("title", "No title")
        link = item.get("url", "")
        highlights = item.get("highlights", [])
        snippet = " ".join(highlights) if highlights else item.get("text", "")[:400]
        formatted.append(f"[{i}] {title}\nURL: {link}\n{snippet}")

    return "\n\n".join(formatted)


In [8]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

web_search_tool = aix.Tool(
    name="Exa Web Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(web_search_exa), "function_name": "web_search_exa"},
)
web_search_tool.save()

code_context_tool = aix.Tool(
    name="Exa Code Context Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(get_code_context_exa), "function_name": "get_code_context_exa"},
)
code_context_tool.save()

company_research_tool = aix.Tool(
    name="Exa Company Research",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(company_research_exa), "function_name": "company_research_exa"},
)
company_research_tool.save()


Tool(path=asma-furniturewala/exa-company-research/aixplain)

# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, three custom script tools wrapping the Exa search API.
* The **instructions** that guide the agent's behaviour and decision-making.

In [10]:
exa_agent = aix.Agent(
    name="Exa Web Search Agent",
    description="A research agent that searches the live web using Exa's neural search API to answer questions, find code examples, and gather company intelligence.",
    instructions="""
    You are a research assistant with access to live web search via Exa's neural search API.
    You have three search tools — use the most appropriate one based on the request:

    - Exa Web Search: for general questions, current news, facts, or any topic that needs
      up-to-date information from the web. Use search_type="deep" for complex research questions.

    - Exa Code Context Search: for programming questions, API usage examples, library documentation,
      or anything involving code. This targets GitHub, Stack Overflow, and official docs.

    - Exa Company Research: for questions about a specific company — its products, funding,
      news, leadership, or competitive landscape.

    Guidelines:
    - Always cite your sources with titles and URLs.
    - Summarise findings clearly and concisely.
    - If one search does not return relevant results, try rephrasing the query.
    - For multi-part questions, call multiple tools and synthesise the results.
    - Never present information as fact without grounding it in a search result.
    """,
    tools=[
        web_search_tool,
        code_context_tool,
        company_research_tool,
    ],
)

print(f"Agent created: {exa_agent.id}")


Agent created: None


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.


In [ ]:
result = exa_agent.run("What are the latest developments in AI agents?")


In [12]:
print(result.data.output)


As of 2023, AI agents have been evolving rapidly, with advancements in natural language processing, machine learning, and automation. Key trends include the integration of AI agents in customer service, personal assistants, and autonomous systems. Companies like OpenAI, Google, and Microsoft are leading the charge, focusing on ethical AI, improved user interaction, and enhanced capabilities. By 2025, we can expect further improvements in AI agents' contextual understanding, emotional intelligence, and their ability to perform complex tasks autonomously.


In [13]:
result = exa_agent.run("Show me how to use the OpenAI function calling API with a Python example.")
print(result.data.output)


To use the OpenAI function calling API in Python, you can follow this example:

import openai

# Set your OpenAI API key
openai.api_key = 'your-api-key'

# Define a function to call
function_call = {
    'name': 'get_weather',
    'description': 'Get the current weather for a given location',
    'parameters': {
        'location': 'string',
        'unit': 'string',
    }
}

# Call the OpenAI API with the function
response = openai.ChatCompletion.create(
    model='gpt-3.5-turbo',
    messages=[
        {'role': 'user', 'content': 'What is the weather like in New York?'},
    ],
    functions=[function_call],
    function_call={'name': 'get_weather'}
)

# Print the response
print(response)

In this example, replace 'your-api-key' with your actual OpenAI API key. The function get_weather is defined with parameters for location and unit. The API is called with a user message, and the response is printed out.


In [14]:
result = exa_agent.run("What is Exa AI and what has the company been up to recently?")
print(result.data.output)


Exa AI is a company focused on artificial intelligence solutions, particularly in data processing and analysis. Recently, they have been involved in developing advanced AI models and applications that enhance data-driven decision-making for businesses. However, specific recent activities or news updates could not be retrieved due to access issues with the necessary data sources.


In [15]:
result = exa_agent.run(
    "Who are their main competitors?",
    session_id=result.data.session_id
)
print(result.data.output)


Exa AI competes with major companies in the artificial intelligence solutions market, including Google AI, Microsoft Azure AI, IBM Watson, and Amazon Web Services (AWS). These companies are known for their advanced AI technologies and data processing capabilities, which position them as significant competitors to Exa AI.


# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.


In [17]:
exa_agent.save()


Agent(path=asma-furniturewala/exa-web-search-agent/aixplain)

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).
